# AfroHealth QA — Experiment 002 (few-shot) on Kaggle or Colab

**Goal:** Answer health questions in the **same language** (Swahili, Luganda, Akan, Amharic, English) using **open multilingual models** with few-shot examples from `Train.csv`.

## Before you run

### Kaggle
1. Create a **Dataset** and upload `Train.csv`, `Test.csv`, `Val.csv` (optional), `SampleSubmission.csv`.
2. Add that dataset to this notebook (**Add data**).
3. **Settings → Accelerator → GPU T4** (or better). **Internet ON** (model download).
4. In the next cell, set `KAGGLE_INPUT_SUBDIR` to the folder name under `/kaggle/input/` (e.g. `afro-health-qa`).

### Google Colab
1. **Runtime → Change runtime type → GPU** (T4 free tier is enough for 4-bit 8B).
2. Put CSVs on Drive, e.g. `MyDrive/afro-health-qa/data/raw/`, or upload to `/content/data/raw/` and set `COLAB_RAW_DIR` below.
3. Run the Drive mount cell if you use Drive.

### Model choice (speed vs. African-language fit)
| Model | Hugging Face id | Notes |
|-------|-----------------|--------|
| **AfriqueLlama-8B** | `McGill-NLP/AfriqueLlama-8B` | Strong fit for competition African languages; Llama-3 chat template. |
| **Aya-Expanse-8B** | `CohereForAI/aya-expanse-8b` | Broad multilingual; good baseline. |

**Faster inference:** keep `USE_BEAM_SEARCH = False` (greedy). **Slightly better quality, ~3–5× slower:** set `USE_BEAM_SEARCH = True` and `NUM_BEAMS = 3`.

If a model is **gated**, add your token: Kaggle **Secrets** `HF_TOKEN`, or Colab **Secrets** / `huggingface-cli login`.

## 1) Configuration — edit this cell only

In [ ]:
import os
from pathlib import Path

# --- Platform detection ---
IS_KAGGLE = os.path.exists("/kaggle/input")
try:
    from google.colab import drive  # noqa: F401

    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print("Kaggle:", IS_KAGGLE, "| Colab:", IS_COLAB)

# --- Model (pick one) ---
MODEL_ID = "McGill-NLP/AfriqueLlama-8B"  # or: "CohereForAI/aya-expanse-8b"

# --- Paths ---
if IS_KAGGLE:
    KAGGLE_INPUT_SUBDIR = "afro-health-qa"  # change to your dataset folder name
    RAW_DIR = Path("/kaggle/input/afro-health-qa") / KAGGLE_INPUT_SUBDIR
    WORK_DIR = Path("/kaggle/working")
else:
    # Colab: either mount Drive and use this, or set to Path("/content/data/raw")
    COLAB_RAW_DIR = Path("/content/drive/MyDrive/afro-health-qa/data/raw")
    RAW_DIR = COLAB_RAW_DIR
    WORK_DIR = Path("/content/exp002_work")

PROCESSED_DIR = WORK_DIR / "processed"
SUBMISSIONS_DIR = WORK_DIR / "submissions"
for d in (PROCESSED_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Generation: time vs quality ---
USE_BEAM_SEARCH = False  # True → slower, sometimes better for medical phrasing
NUM_BEAMS = 3
MAX_NEW_TOKENS = 128  # raise to 192–256 if answers truncate
MIN_NEW_TOKENS = 8
CHECKPOINT_EVERY = 25

# --- Hugging Face token (gated models) ---
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if IS_KAGGLE and not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

## 2) Colab only — mount Google Drive (skip on Kaggle)

In [ ]:
if IS_COLAB and not IS_KAGGLE:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Skipping Drive mount (Kaggle or non-Colab).")

## 3) Install dependencies

Kaggle images usually ship with PyTorch; this cell upgrades/installs **transformers**, **accelerate**, and **bitsandbytes** for 4-bit loading.

In [ ]:
import subprocess
import sys

pkgs = [
    "transformers==4.46.2",
    "accelerate==1.1.1",
    "bitsandbytes==0.44.1",
    "sentencepiece==0.2.0",
    "protobuf==5.28.3",
    "pandas",
    "tqdm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime (Kaggle: GPU T4; Colab: Runtime → GPU). CPU will not run 8B in reasonable time.")

## 4) Load data and build few-shot pools

In [ ]:
import gc
import pandas as pd
from tqdm.auto import tqdm

train_path = RAW_DIR / "Train.csv"
test_path = RAW_DIR / "Test.csv"
sample_path = RAW_DIR / "SampleSubmission.csv"
for p in (train_path, test_path, sample_path):
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Fix RAW_DIR or upload/add dataset.")

train = pd.read_csv(train_path, dtype=str)
test = pd.read_csv(test_path, dtype=str)
sample = pd.read_csv(sample_path, dtype=str)

id_col = "ID"
if "input" in test.columns:
    question_col = "input"
    lang_col = "subset"
    answer_col = "output" if "output" in train.columns else "Target"
else:
    question_col = "Question"
    lang_col = "Language"
    answer_col = "Target" if "Target" in train.columns else "output"


def canonical_prompt_language(lang_code: str) -> str:
    return "eng" if lang_code == "eng" else lang_code


LANG_CODE_MAP = {}
for raw in sorted({str(v) for v in train[lang_col].unique()}):
    key = raw.strip()
    lower = key.lower()
    if lower.startswith("swa") or "swahili" in lower:
        LANG_CODE_MAP[key] = "swa"
    elif lower.startswith("lug") or "luganda" in lower:
        LANG_CODE_MAP[key] = "lug"
    elif lower.startswith("aka") or "twi" in lower or "akan" in lower:
        LANG_CODE_MAP[key] = "aka"
    elif lower.startswith("amh") or "amharic" in lower:
        LANG_CODE_MAP[key] = "amh"
    elif lower.startswith("eng") or "english" in lower:
        LANG_CODE_MAP[key] = "eng"
    else:
        LANG_CODE_MAP[key] = "eng"

FEW_SHOT_EXAMPLES = {}
for raw_lang in train[lang_col].unique():
    canon = LANG_CODE_MAP[str(raw_lang)]
    lang_data = train[train[lang_col] == raw_lang].dropna(subset=[answer_col]).copy()
    lang_data["ans_len"] = lang_data[answer_col].astype(str).str.len()
    ideal = lang_data[(lang_data["ans_len"] >= 50) & (lang_data["ans_len"] <= 300)]
    pool = ideal if len(ideal) >= 2 else lang_data
    n = min(2, len(pool))
    sampled = pool.sample(n, random_state=42) if n else lang_data.sample(min(2, len(lang_data)), random_state=42)
    FEW_SHOT_EXAMPLES[canon] = [(row[question_col], row[answer_col]) for _, row in sampled.iterrows()]

print("Train:", len(train), "Test:", len(test), "Languages:", sorted(set(LANG_CODE_MAP.values())))

SYSTEM_INSTRUCTIONS = {
    "swa": "Wewe ni daktari mtaalamu. Jibu swali la afya hapa chini kwa Kiswahili. Toa jibu kamili na la moja kwa moja. Usiseme huwezi kujibu.",
    "lug": "Oli omusawo omukugu. Ddamu ekibuuzo kino eky'obulamu mu Luganda. Wa eddamu erijjuvu era ery'obwesigwa. Togamba nti toyinza kuddamu.",
    "aka": "Woyɛ dɔkota nimdeɛ. Bua afutuo asɛmmisa yi so wɔ Akan kasa mu. Ma mmuaeɛ a ɛyɛ pɛpɛɛpɛ. Nnka sɛ wontumi mmua so.",
    "amh": "እርስዎ ልምድ ያለው ሐኪም ነዎት። ይህን የጤና ጥያቄ በአማርኛ ይመልሱ። ሙሉ እና ቀጥተኛ መልስ ይስጡ። መመለስ አልችልም አይበሉ።",
    "eng": "You are an expert doctor. Answer the following health question in English. Give a complete and direct answer. Never refuse to answer.",
}

PROMPTS = {
    "swa": "Swali: {question}\nJibu:",
    "lug": "Ekibuuzo: {question}\nEky'okuddamu:",
    "aka": "Asɛmmisa: {question}\nMmuaeɛ:",
    "amh": "ጥያቄ: {question}\nመልስ:",
    "eng": "Question: {question}\nAnswer:",
}

ANSWER_MARKERS = {
    "swa": "Jibu:",
    "lug": "Eky'okuddamu:",
    "aka": "Mmuaeɛ:",
    "amh": "መልስ:",
    "eng": "Answer:",
}

REFUSAL_PATTERNS = [
    "i apologize",
    "i'm sorry",
    "i cannot",
    "i can't",
    "as an ai",
    "as a language model",
    "i'm unable to",
    "i am unable to",
]


def is_refusal(text: str) -> bool:
    lower = text.lower().strip()
    if len(lower) < 5:
        return True
    return any(p in lower for p in REFUSAL_PATTERNS)


def strip_prompt_artefacts(text: str, lang: str) -> str:
    marker = ANSWER_MARKERS.get(lang, "")
    if marker and marker in text:
        text = text.split(marker, 1)[-1]
    text = text.strip()
    if "\n\n" in text:
        text = text.split("\n\n", 1)[0].strip()
    return text


def build_prompt(question: str, lang: str, tokenizer, force_answer: bool = False) -> str:
    sys_msg = SYSTEM_INSTRUCTIONS[lang]
    examples_text = ""
    for ex_q, ex_a in FEW_SHOT_EXAMPLES.get(lang, []):
        examples_text += PROMPTS[lang].format(question=ex_q) + " " + str(ex_a).strip() + "\n\n"
    user_content = examples_text + PROMPTS[lang].format(question=question)
    if force_answer:
        suffix = {
            "swa": " Jibu moja kwa moja bila kusita:",
            "lug": " Ddamu ku biseera bino awatali kwebuuza:",
            "aka": " Bua ntɛm ara:",
            "amh": " አሁን በቀጥታ ይመልሱ:",
            "eng": " Answer directly now:",
        }.get(lang, " Answer directly:")
        user_content += suffix
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": user_content}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            merged = sys_msg + "\n\n" + user_content
            return tokenizer.apply_chat_template(
                [{"role": "user", "content": merged}], tokenize=False, add_generation_prompt=True
            )
    return sys_msg + "\n\n" + user_content

## 5) Load model in 4-bit (fits T4 16GB VRAM)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

for name in ("model", "tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok_kw = {"trust_remote_code": True}
if HF_TOKEN:
    tok_kw["token"] = HF_TOKEN
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **tok_kw)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
model.eval()
device = next(model.parameters()).device
print("Loaded", MODEL_ID, "on", device)

## 6) Generate Test predictions (with resume checkpoint)

In [ ]:
def single_generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    pad = tokenizer.pad_token_id or tokenizer.eos_token_id
    gen_kw = {
        "min_new_tokens": MIN_NEW_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "pad_token_id": pad,
    }
    if USE_BEAM_SEARCH:
        gen_kw["num_beams"] = NUM_BEAMS
        gen_kw["early_stopping"] = True
    else:
        gen_kw["num_beams"] = 1
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kw)
    new_tok = out[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tok, skip_special_tokens=True).strip()


checkpoint_path = PROCESSED_DIR / "predictions_test_exp002_partial.csv"
predictions = []
start_idx = 0
if checkpoint_path.exists():
    partial = pd.read_csv(checkpoint_path, dtype=str).fillna("")
    predictions = partial["prediction"].tolist()
    start_idx = len(predictions)
    print("Resuming from row", start_idx)

for i in tqdm(range(start_idx, len(test)), desc="Test"):
    row = test.iloc[i]
    lang = canonical_prompt_language(LANG_CODE_MAP[str(row[lang_col])])
    q = str(row[question_col])
    prompt = build_prompt(q, lang, tokenizer, force_answer=False)
    answer = single_generate(prompt)
    answer = strip_prompt_artefacts(answer, lang)
    if is_refusal(answer):
        prompt2 = build_prompt(q, lang, tokenizer, force_answer=True)
        answer = single_generate(prompt2)
        answer = strip_prompt_artefacts(answer, lang)
    if is_refusal(answer):
        answer = q
    predictions.append(answer)
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame({"id": test[id_col].astype(str).tolist()[: len(predictions)], "prediction": predictions}).to_csv(
            checkpoint_path, index=False, encoding="utf-8"
        )

pd.DataFrame({"id": test[id_col].astype(str).tolist()[: len(predictions)], "prediction": predictions}).to_csv(
    checkpoint_path, index=False, encoding="utf-8"
)
print("Checkpoint:", checkpoint_path)

## 7) Build Zindi submission CSV

In [ ]:
sub_columns = tuple(sample.columns.tolist())
submission = pd.DataFrame({sub_columns[0]: test[id_col].astype(str).values})
for tc in sub_columns[1:]:
    submission[tc] = predictions

out_name = f"submission_exp002_{MODEL_ID.split('/')[-1].replace('-', '_')}.csv"
submission_path = SUBMISSIONS_DIR / out_name
submission.to_csv(submission_path, index=False, encoding="utf-8")
print("Saved:", submission_path)

if IS_COLAB and not IS_KAGGLE:
    try:
        from google.colab import files

        files.download(str(submission_path))
    except Exception as e:
        print("Download manually:", e)
else:
    print("Kaggle: open Output tab or copy from", submission_path)